# LoRA Fine-tuning en GPU (Google Colab)

**Antes de empezar:**
1. Runtime con GPU activo: *Entorno de ejecución → Cambiar tipo → T4 GPU*
2. Pon la URL HTTPS de tu repo en la celda de configuración

**Orden de ejecución obligatorio:**
1. Ejecuta las celdas **Config → GPU check → Pip install**
2. ⚠️ **REINICIA el runtime** (*Entorno de ejecución → Reiniciar sesión*) — imprescindible para que los paquetes carguen
3. Ejecuta el resto de celdas en orden

In [ ]:
# ── CONFIGURACIÓN ──────────────────────────────────────────────────────────────
GITHUB_REPO_URL = "https://github.com/TU_USUARIO/TU_REPO.git"  # <-- URL HTTPS del repo

# Número de ejemplos. 0 = dataset completo (67k, ~10-15 min en T4)
MAX_TRAIN_SAMPLES = 0
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "GPU no disponible. Ve a Entorno de ejecución → Cambiar tipo de entorno → T4 GPU"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"torch: {torch.__version__}")

In [ ]:
# Colab ya trae PyTorch con CUDA — NO reinstalar torch.
# Instalar solo las dependencias adicionales.
!pip install transformers==5.12.1 peft==0.19.1 datasets==5.0.0 accelerate==1.14.0 \
             transformer-lens==3.4.0 scikit-learn==1.9.0 umap-learn==0.5.12 \
             matplotlib==3.11.0 pyyaml==6.0.3

print()
print("=" * 60)
print("⚠️  REINICIA EL RUNTIME AHORA antes de continuar:")
print("   Entorno de ejecución → Reiniciar sesión")
print("   Después ejecuta desde la celda 'Clone repo' hacia abajo.")
print("=" * 60)

---
## ⚠️ REINICIA EL RUNTIME ANTES DE CONTINUAR
**Entorno de ejecución → Reiniciar sesión**

Después sigue desde la celda siguiente (no hace falta volver a ejecutar pip install).

---

In [ ]:
# Volver a poner la configuración tras el reinicio
GITHUB_REPO_URL = "https://github.com/TU_USUARIO/TU_REPO.git"  # <-- misma URL que arriba
MAX_TRAIN_SAMPLES = 0

In [ ]:
import os
repo_name = GITHUB_REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')

if not os.path.exists(f"/content/{repo_name}"):
    !git clone {GITHUB_REPO_URL} /content/{repo_name}
else:
    print(f"Repo ya existe, actualizando...")
    !git -C /content/{repo_name} pull

%cd /content/{repo_name}
print(f"Directorio: {os.getcwd()}")

In [ ]:
import torch
import transformers
import peft
from transformer_lens import HookedTransformer

print(f"torch:            {torch.__version__}")
print(f"transformers:     {transformers.__version__}")
print(f"peft:             {peft.__version__}")
print(f"CUDA disponible:  {torch.cuda.is_available()}")
print(f"GPU:              {torch.cuda.get_device_name(0)}")

_ = HookedTransformer.from_pretrained('gpt2')
print("TransformerLens:  OK")

In [ ]:
if MAX_TRAIN_SAMPLES and MAX_TRAIN_SAMPLES > 0:
    print(f"Entrenando con {MAX_TRAIN_SAMPLES} ejemplos...")
    !python -m src.train_lora --max-train-samples {MAX_TRAIN_SAMPLES}
else:
    print("Entrenando con el dataset completo (67k ejemplos)...")
    !python -m src.train_lora

In [ ]:
import json, pathlib

result_path = pathlib.Path('results/train_lora.json')
if result_path.exists():
    data = json.loads(result_path.read_text())
    print("=== Resultados del entrenamiento ===")
    for k, v in data.items():
        if not k.startswith('_'):
            print(f"  {k}: {v}")
else:
    print("ERROR: No se encontró results/train_lora.json — el entrenamiento no terminó correctamente.")

In [ ]:
!python -m src.eval --adapter artifacts/lora_adapter

eval_path = pathlib.Path('results/eval_finetuned.json')
if eval_path.exists():
    data = json.loads(eval_path.read_text())
    print(f"\nAccuracy fine-tuned: {data['accuracy']:.4f} ({data['accuracy']*100:.1f}%)")
    print(f"Base accuracy (M0):  0.6170 (61.7%)")
    print(f"Mejora:              {(data['accuracy'] - 0.617)*100:+.1f} pp")

In [ ]:
import shutil, pathlib
from google.colab import files

adapter_dir = pathlib.Path('artifacts/lora_adapter')
if adapter_dir.exists():
    zip_path = '/content/lora_adapter'
    shutil.make_archive(zip_path, 'zip', str(adapter_dir.parent), adapter_dir.name)
    print(f"Descargando {zip_path}.zip ...")
    files.download(f'{zip_path}.zip')
    print("Descomprime en artifacts/lora_adapter/ en tu máquina local.")
else:
    print("ERROR: artifacts/lora_adapter no existe — verifica que el entrenamiento terminó bien.")